#Bronze para Silver.


In [0]:
from config import BRONZE_SCHEMA,SILVER_SCHEMA, CATALOG
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

#### Base de chamados_hora

In [0]:
chamados_hora_bronze = spark.table(f'{BRONZE_SCHEMA}.chamados_hora')
chamados_hora_bronze.printSchema()
display(chamados_hora_bronze)

In [0]:
def normaliza_coluna_data(nome_coluna):
    return F.regexp_replace(F.trim(F.col(nome_coluna)), " .. ", " ")
chamados_hora_silver = (
    chamados_hora_bronze.drop('ingestion_datetime')
    .dropDuplicates() # agora remover a principio somente duplicatas
)
chamados_hora_silver = (
    chamados_hora_silver
    .filter((F.col('ID_Chamado').isNotNull()) & (F.col('ID_Cliente').isNotNull()))
    
    # renomeia as colunas para snake_case
    .withColumnRenamed('ID_Chamado', 'id_chamado')
    .withColumnRenamed('ID_Cliente', 'id_cliente')
    .withColumnRenamed('Hora_Abertura_Chamado', 'hora_abertura_chamado')
    .withColumnRenamed('Hora_Inicio_Atendimento', 'hora_inicio_atendimento')
    .withColumnRenamed('Hora_Finalizacao_Atendimento', 'hora_finalizacao_atendimento')

    # conversão para timestamp
    .withColumn('hora_abertura_chamado', F.to_timestamp(normaliza_coluna_data('hora_abertura_chamado'), format='dd/MM/yyyy HH:mm:ss'))
    .withColumn('hora_inicio_atendimento', F.to_timestamp(normaliza_coluna_data('hora_inicio_atendimento'), format='dd/MM/yyyy HH:mm:ss'))
    .withColumn('hora_finalizacao_atendimento', F.to_timestamp(normaliza_coluna_data('hora_finalizacao_atendimento'), format='dd/MM/yyyy HH:mm:ss'))

    # Preserva a abertura e anula horários fora de ordem, sem excluir chamados.
    .withColumn('hora_inicio_atendimento', F.when(
        F.col('hora_inicio_atendimento') >= F.col('hora_abertura_chamado'),
        F.col('hora_inicio_atendimento')))
    .withColumn('hora_finalizacao_atendimento', F.when(
        F.col('hora_finalizacao_atendimento') >= F.greatest('hora_abertura_chamado', 'hora_inicio_atendimento'),
        F.col('hora_finalizacao_atendimento')))
)
display(chamados_hora_silver)

In [0]:
_,passou = dq_check_unique('chamados_hora_silver', 'id único', chamados_hora_silver, ['id_chamado'])
if not passou:
    raise ValueError("id de chamado duplicado")

_,passou = dq_check('chamados_hora_silver', 'id chamado nulo', chamados_hora_silver, F.col('id_chamado').isNotNull())
if not passou:
    raise ValueError("id de chamado nulo")
_,passou = dq_check('chamados_hora_silver', 'id cliente nulo', chamados_hora_silver, F.col('id_cliente').isNotNull())
if not passou:
    raise ValueError("id de cliente nulo")
_,passou = dq_check('chamados_hora_silver', 'hora de abertura nula', chamados_hora_silver, F.col('hora_abertura_chamado').isNotNull())
if not passou:
    raise ValueError('hora de abertura nula')

In [0]:
chamados_hora_silver.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_SCHEMA}.chamados_hora')

#### Base de clientes

In [0]:
clientes_bronze = spark.table(f'{BRONZE_SCHEMA}.clientes')
clientes_bronze.printSchema()
display(clientes_bronze)

In [0]:
# Formatos de datas presentes na tabela
formatos_datas = ['dd/MM/yyyy', 'yyyy-MM-dd']

clientes_silver = (
    clientes_bronze
    .dropDuplicates(['id_cliente'])
    .filter(F.col('id_cliente').isNotNull())
    
    # Normaliza nomes de pessoas e de regiões
    .withColumn('nome', F.initcap(F.trim(F.col('nome'))))
    .withColumn('regiao', F.initcap(F.trim(F.col('regiao'))))
    .withColumn('email', F.trim(F.col('email')))

    # Coloca idades fora do intervalo de 18 a 110 como null
    .withColumn('idade',
        F.when((F.col('idade') >= 18) & (F.col('idade') <= 110), F.col('idade'))
        .otherwise(F.lit(None)) 
    )

    # Conversão das datas
    .withColumn('data_cadastro', F.coalesce(*[F.try_to_date(F.col('data_cadastro'), f) for f in formatos_datas]))
)

display(clientes_silver)

In [0]:
clientes_silver.groupBy('regiao').count().show()

In [0]:
_,passou = dq_check_unique('clientes_silver', 'id único', clientes_silver, ['id_cliente'])
if not passou:
    raise ValueError("clientes com ids duplicados")
_, passou = dq_check('clientes_silver', 'id único', clientes_silver, F.col('id_cliente').isNotNull())
if not passou :
    raise ValueError("clientes com ids nulos")

In [0]:
clientes_silver.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_SCHEMA}.clientes')